In [ ]:
%%sql -r dataframe_1
use database DB_MONITOR_TOOLS;
use schema PUBLIC;

CREATE OR REPLACE TABLE FINOPS_QUERY_INSIGHTS (
    QUERY_ID STRING,
    USER_NAME STRING,
    WAREHOUSE_NAME STRING,
    TOTAL_ELAPSED_TIME_MINUTES NUMBER(10,2),
    SPILL_OVER_RATIO NUMBER(5,2),
    COMPILATION_RATIO NUMBER(5,2),
    INSIGHT_CATEGORY STRING,
    RECOMMENDATION STRING,
    LOGGED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);


In [ ]:
%%sql -r dataframe_2
-- Deploy the Python code as a native Stored Procedure
CREATE OR REPLACE PROCEDURE SP_RUN_QUERY_FINOPS_AUDIT()
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.10'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'main'
AS
$$
import snowflake.snowpark as snowpark
from snowflake.snowpark.functions import col, lit, when, round, builtin, sql_expr

def main(session: snowpark.Session) -> str:
    # 1. Fetch query history from the account usage shared schema
    query_history_df = session.table("SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY") \
        .filter(col("START_TIME") >= sql_expr("DATEADD(day, -7, CURRENT_TIMESTAMP())")) \
        .filter(col("TOTAL_ELAPSED_TIME") > 10000) 

    # 2. Calculate core FinOps metrics
    metrics_df = query_history_df.select(
        col("QUERY_ID"),
        col("USER_NAME"),
        col("WAREHOUSE_NAME"),
        round(col("TOTAL_ELAPSED_TIME") / 60000, 2).alias("TOTAL_ELAPSED_TIME_MINUTES"),
        round((col("BYTES_SPILLED_TO_LOCAL_STORAGE") + col("BYTES_SPILLED_TO_REMOTE_STORAGE")) / 
              when(col("BYTES_SCANNED") == 0, 1).otherwise(col("BYTES_SCANNED")), 2).alias("SPILL_OVER_RATIO"),
        round(col("COMPILATION_TIME") / 
              when(col("TOTAL_ELAPSED_TIME") == 0, 1).otherwise(col("TOTAL_ELAPSED_TIME")), 2).alias("COMPILATION_RATIO")
    )

    # 3. Apply rule-based categorization logic
    insights_df = metrics_df.filter(
        (col("SPILL_OVER_RATIO") > 0.2) | 
        (col("COMPILATION_RATIO") > 0.4) | 
        (col("TOTAL_ELAPSED_TIME_MINUTES") > 15.0)
    ).select(
        col("QUERY_ID"),
        col("USER_NAME"),
        col("WAREHOUSE_NAME"),
        col("TOTAL_ELAPSED_TIME_MINUTES"),
        col("SPILL_OVER_RATIO"),
        col("COMPILATION_RATIO"),
        when(col("SPILL_OVER_RATIO") > 0.2, lit("DISK_SPILLING"))
        .when(col("COMPILATION_RATIO") > 0.4, lit("COMPILATION_BOTTLENECK"))
        .otherwise(lit("LONG_RUNNING")).alias("INSIGHT_CATEGORY"),
        
        when(col("SPILL_OVER_RATIO") > 0.2, lit("Warehouse out of memory. Upsize warehouse or cluster keys."))
        .when(col("COMPILATION_RATIO") > 0.4, lit("High compile time. Reduce complex metadata or large expression lists."))
        .otherwise(lit("Review query profile for missing filters or Cartesian joins.")).alias("RECOMMENDATION")
    )

    # 4. Save results incrementally
    # FIXED: Import current_timestamp and inject it to match the 9th column (LOGGED_AT)
    from snowflake.snowpark.functions import current_timestamp

    final_payload_df = insights_df.select(
        col("QUERY_ID"), 
        col("USER_NAME"), 
        col("WAREHOUSE_NAME"), 
        col("TOTAL_ELAPSED_TIME_MINUTES"), 
        col("SPILL_OVER_RATIO"), 
        col("COMPILATION_RATIO"), 
        col("INSIGHT_CATEGORY"), 
        col("RECOMMENDATION"),
        current_timestamp().alias("LOGGED_AT") # Matches the target table column name and position
    )
    
    final_payload_df.write.mode("append").save_as_table("FINOPS_QUERY_INSIGHTS")

    return f"Success: Parsed and logged optimization insights."
$$;

-- Automate it to run every morning at 6:00 AM UTC
CREATE OR REPLACE TASK TASK_DAILY_FINOPS_AUDIT
WAREHOUSE = COMPUTE_WH
SCHEDULE = 'USING CRON 0 6 * * * UTC'
AS 
CALL SP_RUN_QUERY_FINOPS_AUDIT();

-- Start the task
ALTER TASK TASK_DAILY_FINOPS_AUDIT RESUME;
